# RetinaNet inference pipeline

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
PIPELINE_DIR = Path('.').resolve()

## TTA, WBF, temporal processing and fusion

In [ ]:
def show_tuning_stage(stage, top=5):
    table = pd.read_csv(PIPELINE_DIR / 'tuning' / f'{stage}.csv').sort_values('mAP', ascending=False)
    fold_columns = [column for column in table.columns if column.startswith('mAP_fold')]
    print(stage, f'({len(table)} settings, top {top})')
    display(table[['key', 'mAP', 'AP50', 'AP75', 'APs'] + fold_columns].head(top).round(4))

In [ ]:
for stage in ['r_wbf', 'r_views', 'r_score', 'r_temporal', 'f_weight', 'f_temporal']:
    show_tuning_stage(stage)

r_wbf (7 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
5,all views: WBF 0.65,0.2355,0.4464,0.2223,0.0792,0.2587,0.2513,0.2482,0.2721,0.1927
6,all views: NMS 0.5 per view + WBF 0.65,0.2355,0.4464,0.2223,0.0792,0.2587,0.2513,0.2482,0.2721,0.1927
3,all views: WBF 0.6,0.2343,0.4469,0.2196,0.0787,0.2576,0.2503,0.2475,0.2713,0.1916
4,all views: NMS 0.5 per view + WBF 0.6,0.2343,0.4469,0.2196,0.0787,0.2576,0.2503,0.2475,0.2713,0.1916
1,all views: WBF 0.55,0.2327,0.4462,0.2168,0.0783,0.2557,0.2489,0.2459,0.2699,0.1903


r_views (8 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
6,flip+none @ 0.75+1.0+1.25+1.5,0.2355,0.4464,0.2223,0.0792,0.2587,0.2513,0.2482,0.2721,0.1927
7,none @ 0.75+1.0+1.25+1.5,0.2296,0.4374,0.2161,0.0773,0.2514,0.2396,0.2514,0.2651,0.1878
4,flip+none @ 0.75+1.0+1.25,0.2285,0.4289,0.2178,0.0665,0.2421,0.2519,0.2454,0.2592,0.1856
5,flip+none @ 1.0+1.25+1.5,0.2250,0.4372,0.2020,0.0793,0.2604,0.2330,0.2367,0.2619,0.1829
3,flip+none @ 1.0+1.5,0.2236,0.4332,0.2030,0.0795,0.2586,0.2306,0.2336,0.2576,0.1815


r_score (4 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
0,score>=0.01,0.2355,0.4464,0.2223,0.0792,0.2587,0.2513,0.2482,0.2721,0.1927
1,score>=0.02,0.2354,0.4464,0.2222,0.0793,0.2587,0.2512,0.2482,0.2721,0.1927
2,score>=0.05,0.2352,0.4462,0.2219,0.0790,0.2591,0.2506,0.2483,0.2718,0.1923
3,conf max,0.2235,0.4265,0.2097,0.0804,0.2515,0.2366,0.2408,0.2571,0.1856


... (2 outputs left out)

f_weight (10 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
3,w_retina 0.25 iou 0.65,0.3001,0.5511,0.2917,0.1075,0.3320,0.3285,0.3200,0.3270,0.2246
5,w_retina 0.5 iou 0.65,0.2999,0.5508,0.2904,0.1084,0.3316,0.3298,0.3194,0.3265,0.2243
0,v2/FCOS ensemble only,0.2976,0.5478,0.2875,0.1051,0.3293,0.3206,0.3179,0.3256,0.2205
7,w_retina 0.75 iou 0.65,0.2975,0.5475,0.2875,0.1082,0.3283,0.3275,0.3167,0.3248,0.2236
9,w_retina 1.0 iou 0.65,0.2944,0.5427,0.2847,0.1076,0.3246,0.3234,0.3134,0.3229,0.2224


f_temporal (20 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
5,s_high 0.2 alpha 0.65 beta 0.6,0.3084,0.5671,0.2979,0.1103,0.3413,0.3386,0.3254,0.3310,0.2329
7,s_high 0.2 alpha 0.8 beta 0.6,0.3083,0.5673,0.2975,0.1106,0.3421,0.3393,0.3255,0.3294,0.2329
6,s_high 0.2 alpha 0.65 beta 1.0,0.3083,0.5668,0.2978,0.1101,0.3417,0.3385,0.3256,0.3315,0.2305
8,s_high 0.2 alpha 0.8 beta 1.0,0.3083,0.5672,0.2973,0.1104,0.3426,0.3393,0.3258,0.3299,0.2311
3,s_high 0.2 alpha 0.5 beta 0.6,0.3080,0.5657,0.2978,0.1101,0.3406,0.3378,0.3253,0.3313,0.2324


In [ ]:
print(json.dumps(json.loads((PIPELINE_DIR / 'tuning/selected.json').read_text()), indent=1))

{
 "wbf_iou": 0.65,
 "pre_nms": null,
 "views": [
  "none@0.75",
  "h@0.75",
  "none@1.0",
  "h@1.0",
  "none@1.25",
  "h@1.25",
  "none@1.5",
  "h@1.5"
 ],
 "score_min": 0.01,
 "conf_type": "avg",
 "retina_temporal": {
  "s_high": 0.2,
  "s_low": 0.01,
  "iou_link": 0.5,
  "max_gap": 2,
  "min_len": 3,
  "alpha": 0.65,
  "beta": 0.6,
  "gamma": 0.75
 },
 "fusion": {
  "weight_fcos": 0.25,
  "iou": 0.65
 },
 "fusion_temporal": {
  "s_high": 0.2,
  "s_low": 0.01,
  "iou_link": 0.5,
  "max_gap": 2,
  "min_len": 3,
  "alpha": 0.65,
  "beta": 0.6,
  "gamma": 0.75
 },
 "v2_fusion": {
  "weight_fcos": 0.75,
  "iou": 0.65
 },
 "v2_fusion_temporal": {
  "s_high": 0.2,
  "s_low": 0.01,
  "iou_link": 0.5,
  "max_gap": 2,
  "min_len": 3,
  "alpha": 0.8,
  "beta": 0.6,
  "gamma": 0.75
 }
}


In [ ]:
official = pd.read_csv(PIPELINE_DIR / 'official_validation_results.csv')
video_columns = [column for column in official.columns if column.startswith('mAP_video_')]
display(official[['key', 'mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR'] + video_columns].round(4))

,key,mAP,AP50,AP75,APs,APm,APl,AR,mAP_video_uav0000086_00000_v,mAP_video_uav0000117_02622_v,mAP_video_uav0000137_00458_v,mAP_video_uav0000182_00000_v,mAP_video_uav0000268_05773_v,mAP_video_uav0000305_00000_v,mAP_video_uav0000339_00001_v
0,Faster R-CNN v2 + temporal (Kaggle 0.3188),0.2841,0.5573,0.2545,0.1560,0.2584,0.4894,0.4996,0.3185,0.3279,0.3603,0.2384,0.1612,0.3553,0.3187
1,Faster R-CNN v2 + FCOS + temporal (best valida...,0.2928,0.5588,0.2692,0.1660,0.2659,0.5005,0.5113,0.3034,0.3357,0.3588,0.2415,0.1922,0.3808,0.3404
2,"RetinaNet 5 checkpoints, single view (trained ...",0.1909,0.3639,0.1786,0.0903,0.1754,0.3552,0.3817,0.2171,0.2386,0.2726,0.1618,0.0952,0.2260,0.2764
3,RetinaNet 5 checkpoints TTA + WBF,0.2249,0.4205,0.2149,0.1161,0.2089,0.3961,0.4606,0.2169,0.2784,0.3148,0.1861,0.1656,0.2911,0.3005
4,RetinaNet + v2/FCOS ensemble fusion (no temporal),0.2720,0.5164,0.2541,0.1355,0.2574,0.4513,0.5111,0.2923,0.3077,0.3542,0.2269,0.1862,0.3763,0.3234
5,RetinaNet 5-fold TTA + WBF + temporal (tuned),0.2419,0.4506,0.2335,0.1170,0.2155,0.4346,0.4610,0.2221,0.3208,0.3170,0.1930,0.1611,0.3076,0.3164
6,RetinaNet + v2/FCOS ensemble fusion + temporal...,0.2883,0.5472,0.2680,0.1713,0.2644,0.4791,0.5114,0.3002,0.3316,0.3583,0.2375,0.1963,0.3791,0.3398


## RetinaNet + Faster R-CNN v2 + temporal (`retina_v2_fusion.py`)

In [ ]:
for stage in ['g_weight_v2', 'g_temporal_v2']:
    show_tuning_stage(stage)

g_weight_v2 (10 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
7,w_retina 0.75 iou 0.65,0.2931,0.5373,0.2846,0.1001,0.3202,0.3240,0.3069,0.3175,0.2202
5,w_retina 0.5 iou 0.65,0.2927,0.5371,0.2835,0.0993,0.3197,0.3223,0.3066,0.3178,0.2205
9,w_retina 1.0 iou 0.65,0.2921,0.5355,0.2829,0.1005,0.3190,0.3235,0.3061,0.3169,0.2196
3,w_retina 0.25 iou 0.65,0.2897,0.5337,0.2795,0.0973,0.3169,0.3168,0.3045,0.3163,0.2211
6,w_retina 0.75 iou 0.55,0.2886,0.5323,0.2780,0.1001,0.3165,0.3191,0.3025,0.3139,0.2151


g_temporal_v2 (20 settings, top 5)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
7,s_high 0.2 alpha 0.8 beta 0.6,0.3011,0.5543,0.2908,0.1030,0.3334,0.3352,0.3177,0.3245,0.2305
5,s_high 0.2 alpha 0.65 beta 0.6,0.3008,0.5539,0.2905,0.1031,0.3330,0.3341,0.3164,0.3239,0.2295
8,s_high 0.2 alpha 0.8 beta 1.0,0.3007,0.5534,0.2903,0.1024,0.3331,0.3352,0.3174,0.3243,0.2272
6,s_high 0.2 alpha 0.65 beta 1.0,0.3003,0.5528,0.2900,0.1026,0.3327,0.3340,0.3161,0.3237,0.2269
3,s_high 0.2 alpha 0.5 beta 0.6,0.3001,0.5525,0.2902,0.1030,0.3319,0.3326,0.3147,0.3231,0.2284


In [ ]:
decision = json.loads((PIPELINE_DIR / 'retina_v2_fusion_decision.json').read_text())
display(pd.DataFrame(decision['per_video'], index=['RetinaNet + v2 + temporal', 'A: v2 + temporal']).T)
print({key: value for key, value in decision.items() if key != 'per_video'})

,RetinaNet + v2 + temporal,A: v2 + temporal
video_uav0000086_00000_v,0.3063,0.3185
video_uav0000117_02622_v,0.3629,0.3279
video_uav0000137_00458_v,0.3578,0.3603
video_uav0000182_00000_v,0.2327,0.2384
video_uav0000268_05773_v,0.1878,0.1612
video_uav0000305_00000_v,0.3574,0.3553
video_uav0000339_00001_v,0.3365,0.3187


{'mAP': 0.28420616002718846, 'reference_A': 0.28410413783410143, 'gain': 0.00010202219308702576, 'videos_improved': 4, 'videos': 7, 'adopt': False}
